# Part 2.B — Fine-tuning Pretrained GPT2 and BERT

Part 2.A trained a joint model from scratch. Here we fine-tune **pretrained** models on
the same two ATIS tasks in a **multi-task** setting, and compare a decoder-only model
against an encoder-only one:

| Model | Type | Intent representation |
|-------|------|-----------------------|
| **GPT2** (`openai-community/gpt2`) | decoder-only, causal | hidden state of the **last** real token — only that position has seen the whole utterance |
| **BERT** (`google-bert/bert-base-uncased`) | encoder-only, bidirectional | the **`[CLS]`** token, which is designed as a sentence-level summary |

**Metrics**
- Intent classification: **accuracy**
- Slot filling: **F1 score** via `conll.evaluate`

## The sub-tokenization problem

ATIS labels are **one slot tag per word**, but both tokenizers split words into subword
pieces, so the number of tokens no longer matches the number of labels:

```
word    : washington                  ->  wash ##ing ##ton
slot    : B-fromloc.city_name         ->  ???  ???   ???
```

The standard fix (Chen et al., 2019, https://arxiv.org/abs/1902.10909) is to align labels
to the **first sub-token of each word** and ignore the rest in the loss, using a label of
`-100` so `CrossEntropyLoss` skips them. At prediction time only first-sub-token
positions are read back, restoring one tag per original word.

## 1. Setup

In [ ]:
import os
import json
import math
import copy
import random
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.utils.data import DataLoader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

from transformers import AutoTokenizer, AutoModel, AutoConfig

IGNORE_INDEX = -100   # CrossEntropyLoss skips these positions

## 2. Data

In [ ]:
# conll.py provides the official CoNLL evaluation used for slot F1
import sys
sys.path.append('.')
from conll import evaluate

PAD_TOKEN = 0


def load_json(path):
    with open(path) as f:
        return json.load(f)


tmp_train = load_json('dataset/ATIS/train.json')
test_raw  = load_json('dataset/ATIS/test.json')

# ATIS ships no dev split, so carve one out of train, stratified on intent.
# Intents occurring only once cannot be split and stay in train.
intent_counts = Counter(x['intent'] for x in tmp_train)
single, multi = [], []
for x in tmp_train:
    (single if intent_counts[x['intent']] == 1 else multi).append(x)

from sklearn.model_selection import train_test_split
train_raw, dev_raw = train_test_split(
    multi, test_size=0.10, random_state=SEED, shuffle=True,
    stratify=[x['intent'] for x in multi])
train_raw += single

print(f'Train: {len(train_raw)} | Dev: {len(dev_raw)} | Test: {len(test_raw)}')
print(f'Intents: {len(set(x["intent"] for x in tmp_train))}')
print('\nExample:')
print(json.dumps(train_raw[0], indent=1))

### Label vocabularies

Only the slot and intent label spaces are needed now — the pretrained tokenizers supply
the word vocabulary.

In [ ]:
slot2id   = {'pad': 0}
for s in sorted({s for x in train_raw for s in x['slots'].split()}):
    slot2id.setdefault(s, len(slot2id))
intent2id = {v: i for i, v in enumerate(sorted({x['intent'] for x in train_raw}))}
id2slot   = {v: k for k, v in slot2id.items()}
id2intent = {v: k for k, v in intent2id.items()}

print(f'Slots: {len(slot2id)} | Intents: {len(intent2id)}')

### Sub-token alignment

> **TODO** — this is the core of Part 2.B. Use `tokenizer(words, is_split_into_words=True)`
> and `encoding.word_ids()` to map each sub-token back to its source word. Assign the
> word's slot label to the **first** sub-token and `IGNORE_INDEX` to every continuation
> sub-token and special token.

In [ ]:
def align_labels(words, slots, tokenizer, max_len=64):
    # TODO:
    #   enc = tokenizer(words, is_split_into_words=True, truncation=True,
    #                   max_length=max_len)
    #   word_ids = enc.word_ids()
    #   label = slot2id[slots[wid]] if this is the first sub-token of word wid
    #           else IGNORE_INDEX
    # returns: input_ids, attention_mask, aligned_labels
    raise NotImplementedError


def make_loader(raw, tokenizer, batch_size=32, shuffle=False):
    # TODO: build a DataLoader yielding input_ids, attention_mask, slot_labels,
    #       intent_label, padded within each batch
    raise NotImplementedError

## 3. Models

One joint head design per backbone. The slot head is shared in shape — a per-token
classifier — while the intent head differs because the two architectures expose sentence
meaning at different positions.

> **TODO** — complete both classes.

In [ ]:
class JointBERT(nn.Module):
    # encoder-only: [CLS] carries the sentence representation
    def __init__(self, name='google-bert/bert-base-uncased', n_slots=None,
                 n_intents=None, dropout=0.1):
        super().__init__()
        # TODO: self.bert = AutoModel.from_pretrained(name)
        # TODO: dropout, slot_head (hidden -> n_slots), intent_head (hidden -> n_intents)
        raise NotImplementedError

    def forward(self, input_ids, attention_mask):
        # TODO: out = self.bert(...); sequence = out.last_hidden_state
        #       intent from sequence[:, 0]  ([CLS]); slots from every position
        raise NotImplementedError


class JointGPT2(nn.Module):
    # decoder-only: use the LAST non-padding token, the only one that has seen everything
    def __init__(self, name='openai-community/gpt2', n_slots=None, n_intents=None,
                 dropout=0.1):
        super().__init__()
        # TODO: self.gpt2 = AutoModel.from_pretrained(name)
        #       remember: GPT2 has no pad token -> tokenizer.pad_token = eos_token
        raise NotImplementedError

    def forward(self, input_ids, attention_mask):
        # TODO: last_idx = attention_mask.sum(1) - 1
        #       intent from sequence[arange(B), last_idx]; slots from every position
        raise NotImplementedError

## 4. Training and evaluation

Joint loss is the sum of intent and slot cross-entropies; slot loss uses
`ignore_index=IGNORE_INDEX` so continuation sub-tokens and specials are skipped.

> **TODO** — at evaluation, read predictions back at first-sub-token positions only, so
> the output is one tag per original word and `conll.evaluate` can be applied.

In [ ]:
def train_loop(loader, optimizer, model, criterion_slots, criterion_intents, clip=1.0):
    raise NotImplementedError


def eval_loop(loader, model, criterion_slots, criterion_intents, raw):
    # returns slot_f1, intent_accuracy, loss
    raise NotImplementedError


def run_experiment(name, model_fn, lr, n_epochs=5, patience=3):
    # TODO: mirror Part 2.A's runner; append to results
    raise NotImplementedError

---
## Experiments

Fine-tune both backbones and compare. Pretrained models need a much smaller learning
rate than the from-scratch model in Part 2.A — `5e-5` to `2e-5` is the usual range.

In [ ]:
results = []

for lr in [5e-5, 3e-5, 2e-5]:
    run_experiment(f'B_BERT_lr_{lr:g}',  model_fn=lambda: JointBERT(...),  lr=lr)
    run_experiment(f'B_GPT2_lr_{lr:g}',  model_fn=lambda: JointGPT2(...),  lr=lr)

---
## Results Summary

In [ ]:
df = pd.DataFrame(results)
display(df)

for family in ['BERT', 'GPT2']:
    rows = [r for r in results if family in r['name']]
    if rows:
        best = max(rows, key=lambda r: r['dev_f1'])
        print(f"{family:5s} best: {best['name']}  "
              f"test slot F1 {best['test_f1']:.4f} | intent acc {best['test_acc']:.4f}")